In [1]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/omarkhashab/club-signups-csv/club_signups.csv')
# df = pd.read_csv("club_signups.csv")

print("Shape:")
print(df.shape)

print("\nData types:")
print(df.dtypes)

for col in ["faculty", "club", "city", "fee_paid"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

Shape:
(39, 9)

Data types:
signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object

faculty:
faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
iet                                       3
MET                                       3
Engineering and Materials Science         1
Pharma                                    1
Media Engineering and Technology          1
met                                       1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64

club:
club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Football       3
Chess Club     3
Socc

In [2]:
# 1. Normalize text
for col in ["faculty", "club", "city"]:
    df[col] = df[col].str.strip().str.lower()

# 2. Create mapping dictionaries
faculty_map = {
    "met": "MET",
    "media engineering and technology": "MET",
    "iet": "IET",
    "information engineering and technology": "IET",
    "ems": "EMS",
    "engineering and materials science": "EMS",
    "pharmacy": "Pharmacy",
    "pharma": "Pharmacy",
    "management": "Management"
}

club_map = {
    "robotics": "Robotics",
    "debate": "Debate",
    "debating": "Debate",
    "debate club": "Debate",
    "music": "Music",
    "football": "Football",
    "soccer": "Football",
    "chess": "Chess",
    "chess club": "Chess"
}

city_map = {
    "cairo": "Cairo",
    "giza": "Giza",
    "el giza": "Giza",
    "alexandria": "Alexandria",
    "alex": "Alexandria"
}

# 3. Apply mappings
df["faculty"] = df["faculty"].map(faculty_map)
df["club"] = df["club"].map(club_map)
df["city"] = df["city"].map(city_map)

# 4. Assert that all values are canonical
assert df["faculty"].isin(["MET", "IET", "EMS", "Pharmacy", "Management"]).all()
assert df["club"].isin(["Robotics", "Debate", "Music", "Football", "Chess"]).all()
assert df["city"].isin(["Cairo", "Giza", "Alexandria"]).all()

# 5. Show cleaned values
for col in ["faculty", "club", "city"]:
    print(f"\n{col}:")
    print(df[col].value_counts())

print("\nAll categorical values are valid!")


faculty:
faculty
IET           12
Pharmacy       9
EMS            9
MET            5
Management     4
Name: count, dtype: int64

club:
club
Football    9
Chess       9
Music       8
Robotics    8
Debate      5
Name: count, dtype: int64

city:
city
Alexandria    16
Cairo         16
Giza           7
Name: count, dtype: int64

All categorical values are valid!


In [3]:
# Clean names
df["full_name"] = (
    df["full_name"]
    .str.strip()
    .str.split()
    .str.join(" ")
    .str.title()
)

# Clean emails
df["email"] = df["email"].str.strip().str.lower()

# Show results
print(df[["full_name", "email"]].head(10))

       full_name                             email
0      Hana Zaki      hana.zaki@student.guc.edu.eg
1     Ahmed Zaki     ahmed.zaki@student.guc.edu.eg
2   Mohamed Adel   mohamed.adel@student.guc.edu.eg
3   Mohamed Adel   mohamed.adel@student.guc.edu.eg
4  Laila Mostafa  laila.mostafa@student.guc.edu.eg
5    Ali Ibrahim    ali.ibrahim@student.guc.edu.eg
6     Ahmed Adel     ahmed.adel@student.guc.edu.eg
7      Omar Adel      omar.adel@student.guc.edu.eg
8    Ali Ibrahim    ali.ibrahim@student.guc.edu.eg
9    Ali Ibrahim    ali.ibrahim@student.guc.edu.eg


In [4]:
# 1. Normalize values
df["fee_paid"] = df["fee_paid"].astype(str).str.strip().str.lower()

# 2. Create a mapping dictionary
fee_map = {
    "yes": True,
    "y": True,
    "true": True,
    "1": True,
    "no": False,
    "n": False,
    "false": False,
    "0": False
}

# 3. Apply mapping
df["fee_paid"] = df["fee_paid"].map(fee_map)

# 4. Check for unmapped values
assert df["fee_paid"].notna().all()

# 5. Convert to boolean dtype
df["fee_paid"] = df["fee_paid"].astype(bool)

# 6. Verify results
print(df["fee_paid"].value_counts())
print(df["fee_paid"].dtype)

fee_paid
True     27
False    12
Name: count, dtype: int64
bool


In [5]:
print(df["signed_up_at"].unique())

['2026-09-15 13:39' '15/09/2026 03:12' '2026-09-14 23:32'
 '2026-09-17 11:15' '2026-09-13 22:36' '15/09/2026 03:42'
 '2026-09-14 17:03' '2026-09-13 23:52' '2026-09-15 23:07'
 '2026-09-16 09:07' '2026-09-13 21:16' '14/09/2026 01:22'
 '16/09/2026 23:55' '2026-09-16 20:31' '13/09/2026 15:38'
 '2026-09-16 01:39' '2026-09-17 00:41' '2026-09-13 14:36'
 '2026-09-16 16:16' '2026-09-17 17:14' '2026-09-16 09:29'
 '2026-09-17 23:36' '16/09/2026 01:15' '2026-09-15 01:05'
 '16/09/2026 02:01' '17/09/2026 12:27' '15/09/2026 17:20'
 '15/09/2026 20:20' '15/09/2026 18:40' '15/09/2026 03:28'
 '2026-09-15 11:36' '2026-09-17 23:29' '15/09/2026 18:01'
 '2026-09-17 06:54' '2026-09-15 05:43' '18/09/2026 23:36']


In [6]:
# Identify dates containing hyphens
format1 = df["signed_up_at"].str.contains("-", regex=False)

# Parse both formats separately
dates1 = pd.to_datetime(
    df.loc[format1, "signed_up_at"],
    format="%Y-%m-%d %H:%M"
)

dates2 = pd.to_datetime(
    df.loc[~format1, "signed_up_at"],
    format="%d/%m/%Y %H:%M"
)

# Combine into one datetime column
df.loc[format1, "signed_up_at"] = dates1
df.loc[~format1, "signed_up_at"] = dates2

df["signed_up_at"] = pd.to_datetime(df["signed_up_at"])

# Verify
assert df["signed_up_at"].notna().all()

print(df["signed_up_at"].dtype)
print(df["signed_up_at"].head(10))

datetime64[ns]
0   2026-09-15 13:39:00
1   2026-09-15 03:12:00
2   2026-09-14 23:32:00
3   2026-09-17 11:15:00
4   2026-09-13 22:36:00
5   2026-09-15 03:42:00
6   2026-09-14 17:03:00
7   2026-09-13 23:52:00
8   2026-09-15 23:07:00
9   2026-09-16 09:07:00
Name: signed_up_at, dtype: datetime64[ns]


In [7]:
# 1. Count rows before removing duplicates
before = len(df)

# 2. Count exact duplicates
exact_duplicates = df.duplicated().sum()

print("Exact duplicates:", exact_duplicates)

# 3. Remove exact duplicates
df = df.drop_duplicates()

# 4. Count remaining rows
after = len(df)

print("Rows before:", before)
print("Rows removed:", before - after)
print("Rows remaining:", after)

Exact duplicates: 3
Rows before: 39
Rows removed: 3
Rows remaining: 36


In [8]:
# 1. Count rows before
before = len(df)

# 2. Sort by submission date (oldest to newest)
df = df.sort_values("signed_up_at")

# 3. Keep the latest submission per student per club
df = df.drop_duplicates(
    subset=["student_id", "club"],
    keep="last"
)

# 4. Count removed rows
removed = before - len(df)

print("Repeated sign-ups removed:", removed)
print("Rows remaining:", len(df))

# 5. Verify student-club combinations are unique
assert not df.duplicated(subset=["student_id", "club"]).any()

print("All student-club combinations are unique!")

Repeated sign-ups removed: 4
Rows remaining: 32
All student-club combinations are unique!


In [9]:
df.to_csv("/kaggle/working/club_signups_clean.csv", index=False)